# E2 — Lịch trình learning rate ± warm-up (yêu cầu 5.1.2)

**Hướng dẫn người chạy:** bật GPU **T4** (không chọn P100) và Internet → sửa `OWNER` → *Save & Run All* (chạy nền) → tải `outputs/` (file `.zip` in ở cuối) gửi cho điều phối. Không sửa gì khác.

Optimizer SGD momentum 0.9, lr 0.1, wd 5e-4, batch 128, 20 epoch (7 020 bước). Mọi lịch được cập nhật **theo bước**. 6 cấu hình × 2 seed = 12 lượt. Notebook này ghi thêm **loss theo từng bước** (`steps.csv`) để vẽ đường loss theo bước.

| config | lịch | warm-up |
|---|---|---|
| E2-const / E2-const-wu | hằng số 0.1 | không / có |
| E2-step / E2-step-wu | ×0.1 tại epoch 10 và 15 | không / có |
| E2-cos / E2-cos-wu | cosine 0.1 → 0 | không / có |

## Giải thích

- **Step decay:** giữ LR không đổi rồi nhân 0.1 tại các mốc. Ở đây mốc ở 50% và 75% tổng ngân sách, tức bước 3 510 (epoch 10) và 5 265 (epoch 15); LR là 0.1 → 0.01 → 0.001. Loss thường rơi thành bậc tại mốc vì bước nhảy hết nhiễu của LR lớn.
- **Cosine annealing:** LR giảm mượt theo nửa đường cosine từ 0.1 về 0 trong suốt quá trình (có warm-up thì cosine bắt đầu sau warm-up).
- **Warm-up:** LR tăng tuyến tính từ `0.01 × lr` lên `lr` trong 2 epoch đầu (702 bước); mốc step decay vẫn tính trên tổng 20 epoch. Mục đích: tránh bước quá lớn khi trọng số/thống kê chuẩn hóa còn chưa ổn định.
- **Giả thuyết cần kiểm chứng (chưa phải kết quả):** step có bậc rơi loss tại mốc decay; const dao động ở mức cao; ở lr 0.1 có BatchNorm thì warm-up nhiều khả năng nằm trong nhiễu, và đó là kết luận hợp lệ.

In [1]:
# ===== CHỈ SỬA CÁC DÒNG TRONG KHỐI NÀY =====
OWNER = "ten_ban"          # tên của bạn (không dấu, không khoảng trắng)
SEEDS = [0, 1]
# ============================================

import os, sys
REPO_URL = "https://github.com/minh301003/Deep-Learning-Project-2026B.git"
TAG = "v1"                 
if not os.path.exists("repo"):
    !git clone --depth 1 --branch $TAG $REPO_URL repo
sys.path.insert(0, "repo")
import pandas as pd
import btl
btl.init(owner=OWNER)

In [ ]:
RUNS = {
    "E2-const":    dict(schedule="const"),
    "E2-step":     dict(schedule="step"),
    "E2-cos":      dict(schedule="cosine"),
    "E2-const-wu": dict(schedule="const",  warmup_epochs=2),
    "E2-step-wu":  dict(schedule="step",   warmup_epochs=2),
    "E2-cos-wu":   dict(schedule="cosine", warmup_epochs=2),
}

for seed in SEEDS:
    for name, overrides in RUNS.items():
        btl.run(name, seed, log_steps=True, **overrides)

In [ ]:
runs = pd.read_csv(f"{btl.OUT_DIR}/runs.csv")
print(runs[["config", "seed", "status", "final_test_acc", "duration_min"]].to_string(index=False))
bad = runs[runs.status != "completed"]
btl.zip_outputs("02_lr_schedules")